# 02 — TSP QUBO + Classical Baselines

Phase 2 のノートブック。巡回セールスマン問題 (TSP) を QUBO に変換し、
古典ベースライン (brute force, simulated annealing) で解く。

定式化は Lucas (2014) §6.4 に従う。n 都市で n² 変数 x[i,t] (city i が時刻 t に
訪問されるとき 1)、index 規則は `i*n + t` (city-major)。

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

from quantum_optimizer import (
    brute_force,
    decode_tsp_tour,
    simulated_annealing,
    tsp_to_qubo,
)

rng = np.random.default_rng(0)

In [ ]:
def random_tsp_instance(n: int, seed: int) -> tuple[np.ndarray, np.ndarray]:
    coords = np.random.default_rng(seed).random((n, 2))
    D = np.linalg.norm(coords[:, None] - coords[None, :], axis=-1)
    return coords, D


coords4, D4 = random_tsp_instance(4, seed=1)
qubo4 = tsp_to_qubo(D4)
print(f"QUBO shape: {qubo4.Q.shape}, non-zero off-diagonal: "
      f"{int(np.count_nonzero(np.triu(qubo4.Q, k=1)))}")
print(f"offset = {qubo4.offset:.3f}")

In [ ]:
res = brute_force(qubo4)
sol = decode_tsp_tour(res.bitstring, 4, D4)
print(f"bitstring energy = {res.energy:.4f}, evaluated {res.num_evals} states")
print(f"tour = {sol.tour}, distance = {sol.total_distance:.4f}, valid = {sol.is_valid}")

In [ ]:
coords8, D8 = random_tsp_instance(8, seed=2)
qubo8 = tsp_to_qubo(D8)
sa = simulated_annealing(qubo8, num_reads=20, num_sweeps=2000, seed=0)
sol8 = decode_tsp_tour(sa.bitstring, 8, D8)
print(f"SA best energy = {sa.energy:.4f}")
print(f"tour = {sol8.tour}, distance = {sol8.total_distance:.4f}, valid = {sol8.is_valid}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for k, hist in enumerate(sa.meta["histories"]):
    ax.plot(hist, alpha=0.5, label=f"read {k}" if k < 3 else None)
ax.set_xlabel("sweep")
ax.set_ylabel("energy")
ax.set_title("SA energy traces (8-city TSP, 20 reads)")
ax.legend(loc="upper right")
fig.tight_layout()

In [ ]:
G = nx.Graph()
for i, (x, y) in enumerate(coords8):
    G.add_node(i, pos=(float(x), float(y)))
tour_edges = [
    (sol8.tour[k], sol8.tour[(k + 1) % len(sol8.tour)])
    for k in range(len(sol8.tour))
]
G.add_edges_from(tour_edges)

fig, ax = plt.subplots(figsize=(5, 5))
pos = nx.get_node_attributes(G, "pos")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#4c72b0", node_size=350)
nx.draw_networkx_labels(G, pos, ax=ax, font_color="white")
nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#dd8452", width=2)
ax.set_title(f"8-city tour (distance = {sol8.total_distance:.3f})")
ax.set_aspect("equal")
fig.tight_layout()

In [ ]:
# Penalty sensitivity: small A makes constraint-violating bitstrings cheaper than
# valid tours, so the solver returns infeasible solutions.
coords5, D5 = random_tsp_instance(5, seed=3)
default_A = float(D5.max()) * 5 + 1.0
ratios = [0.01, 0.1, 0.5, 1.0, 2.0]
for r in ratios:
    qubo = tsp_to_qubo(D5, penalty=r * default_A)
    res = simulated_annealing(qubo, num_reads=10, num_sweeps=1000, seed=0)
    sol = decode_tsp_tour(res.bitstring, 5, D5)
    print(f"A/default = {r:>5.2f}  valid = {sol.is_valid}  distance = {sol.total_distance:.3f}")

## 次フェーズ

Phase 3 では同じ `QuboProblem` を QAOA on Braket `LocalSimulator` に乗せる。
`QuboProblem.to_ising()` で得た `(h, J, const)` から QAOA ハミルトニアンを
構築する想定。